# MadmaxRuns paper figures

This notebook renders study results through `MadmaxRunsPlots.make_all`, the same entry point used by the command-line exporter. The repository defines the figure geometry, colors, fonts, LaTeX labels, and PDF and PNG export settings.

Run `julia scripts/setup.jl notebooks` from the repository root and select **MadmaxRuns (Julia)**. Install the DejaVu Serif font family for full paper export. Open the notebook from the repository root.

In [ ]:
repo = pwd()
isfile(joinpath(repo, "plots", "Project.toml")) || error("Open this notebook from the MadmaxRuns repository root")
output_root = get(ENV, "MADMAX_OUT", joinpath(repo, "results"))
generate_studies = false
quick = false
require_all = true
study_overrides = Dict{String,NamedTuple}()

import Pkg
Pkg.activate(joinpath(repo, "plots"))
using MadmaxRuns, MadmaxRunsPlots
paths = prepare_output_layout(output_root)
paths

## Study data

By default the notebook reads the `.jls` study results already in `results/data`. Set `generate_studies = true` and rerun the settings cell to compute them with the paper study drivers. Completed compatible studies are reused. Set `quick = true` to test with fewer simulations. That changes the plotted data.

A full run can take substantial time. If a scan point gives an invalid analytical probability, its log and saved statistics identify where it stopped. You can choose study options in the settings cell, for example `study_overrides["pvalues"] = (; taus = [0.01])` for a reduced scan. Set `require_all = false` to preview figures from the available studies.

In [ ]:
if generate_studies
    with_run_log(joinpath(paths.logs, "paperplot_studies.log")) do
        configure_threads()
        for name in STUDY_ORDER
            name in PAPER_PLOT_STUDIES || continue
            options = get(study_overrides, name, (;))
            run_study(name; outdir = paths.data, resume = true, quick, options...)
        end
    end
end

## Export figures

The renderer writes PDF and PNG pairs to `results/paperplots` and appends progress to `results/logs/paperplots.log`.

In [ ]:
made = with_run_log(joinpath(paths.logs, "paperplots.log")) do
    log_message("Rendering studies from $(paths.data) to $(paths.paperplots)")
    figures = make_all(paths.data, paths.paperplots; require_all)
    isempty(figures) && error("No paper figures were rendered. Generate study data first.")
    log_message("$(length(figures)) figures written to $(paths.paperplots)")
    figures
end